## Predicting Life Expectancy Across U.S. Counties Using Socioeconomic, Healthcare, Environmental, and Community Indicators

In [1]:
import pandas as pd

path = "2025 County Health Rankings Data_.xlsx"

df = pd.read_excel(path)
df.head()


,FIPS,State,County,Unreliable,Deaths,Years of Potential Life Lost Rate,95% CI - Low,95% CI - High,National Z-Score,YPLL Rate (Hispanic (all races)),...,95% CI - Low.42,95% CI - High.42,# Children in Single-Parent Households,# Children in Households,% Children in Single-Parent Households,95% CI - Low.43,95% CI - High.43,# Rural Residents,% Rural,Population
0,1000.0,Alabama,NaN,NaN,102760.0,11853.247248,11744.014820,11962.479676,NaN,NaN,...,1.057282,1.181115,344866.0,1125290.0,30.646855,29.977972,31.315739,2123399.0,42.262760,5108468.0
1,1001.0,Alabama,Autauga,NaN,1008.0,9938.263382,9021.297133,10855.229632,-0.144182,NaN,...,0.000000,0.804629,3099.0,13921.0,22.261332,17.559107,26.963556,23920.0,40.676813,60342.0
2,1003.0,Alabama,Baldwin,NaN,3944.0,8957.112686,8499.339223,9414.886148,-0.400274,4636.48424,...,0.550919,1.122949,10022.0,51059.0,19.628273,17.032801,22.223745,87113.0,37.586455,253507.0
3,1005.0,Alabama,Barbour,NaN,587.0,12738.656137,11133.499025,14343.813250,0.586752,NaN,...,0.523561,2.710387,2680.0,5160.0,51.937984,43.805069,60.070900,16627.0,65.919994,24585.0
4,1007.0,Alabama,Bibb,NaN,509.0,11708.948038,10166.716823,13251.179253,0.317987,NaN,...,0.000000,1.468004,1383.0,4430.0,31.218962,23.551770,38.886153,22293.0,100.000000,21868.0


In [2]:
df.shape

(3211, 618)

In [3]:
def drop_missing_columns(df, threshold=0.10):
    """
    Drops all columns where the percentage of missing values 
    is greater than the given threshold (default = 40%).
    """
    missing_fraction = df.isna().mean()       # % missing per column
    cols_to_keep = missing_fraction[missing_fraction <= threshold].index
    return df[cols_to_keep]

In [4]:
df = drop_missing_columns(df)

In [5]:
import pandas as pd
from sklearn.model_selection import train_test_split

# --- Select your response variable ---
target = "Life Expectancy"

# --- Drop rows with missing target ---
df = df.dropna(subset=[target]).reset_index(drop=True)

# --- Create quantile bins for stratified split ---
df["strata"] = pd.qcut(df[target], q=10, duplicates='drop')

# ---- First split: Train (70%) vs Temp (30%) ----
train_df, temp_df = train_test_split(
    df, 
    test_size=0.30, 
    random_state=42, 
    stratify=df["strata"]
)

# ---- Second split Temp → Eval (15%) + Holdout (15%) ----
eval_df, holdout_df = train_test_split(
    temp_df, 
    test_size=0.50, 
    random_state=42,
    stratify=temp_df["strata"]
)

# Drop the artificial stratification column
for d in [train_df, eval_df, holdout_df]:
    d.drop(columns=["strata"], inplace=True)

# --- Save the files ---
train_df.to_csv("data/raw/train.csv", index=False)
eval_df.to_csv("data/raw/eval.csv", index=False)
holdout_df.to_csv("data/raw/holdout.csv", index=False)



print("Train shape:", train_df.shape)
print("Eval shape:", eval_df.shape)
print("Holdout shape:", holdout_df.shape)


Train shape: (2183, 232)
Eval shape: (468, 232)
Holdout shape: (468, 232)
